# 02 — Build Leave-One-Generator-Out (LOGO) Splits

Builds the 4 cross-generator rotations the whole experiment runs on. For each rotation `r`:
- **hold out** generator `r` entirely — its reviews become the *cross-generator* test set
- **train** on human + the other 3 generators
- **in-dist val/test** are held-out portions of those 3 training generators + matched human

Critically, the human reviews are split into a **disjoint pool**: the 2,500 humans paired with the held-out generator are never seen in training, so cross-gen evaluation isn't contaminated by familiar human text.

**Inputs** (from notebooks 01 / 01b / 01c, on Drive):
- `data/raw/human_reviews.csv` — ~10,000 human reviews (`label=0`, `generator='human'`)
- `data/generated/ai_reviews.csv` — shared file, ~10,000 AI reviews tagged by `generator`

**Output:** `data/splits/rotation_{0..3}/{train,val_indist,test_indist,test_crossgen}.csv` + `meta.csv`

CPU-only, runs in seconds. Run `00_setup` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ECS111FinalProject'
HUMAN_PATH = os.path.join(PROJECT_DIR, 'data', 'raw', 'human_reviews.csv')
AI_PATH = os.path.join(PROJECT_DIR, 'data', 'generated', 'ai_reviews.csv')
SPLITS_DIR = os.path.join(PROJECT_DIR, 'data', 'splits')
os.makedirs(SPLITS_DIR, exist_ok=True)
%cd $PROJECT_DIR

## Config

`GENERATORS` is the rotation order — rotation `r` holds out `GENERATORS[r]`. `granite` from the original proposal was swapped for `deepseek` because no Granite model is served through the HF router (see notebook 01c).

In [ ]:
GENERATORS = ['gpt5mini', 'deepseek', 'gemma', 'qwen']
SEED = 42
N_CROSSGEN_HUMAN = 2500   # human reviews reserved per rotation for the cross-gen test set

## Load data and split the shared AI file by generator

`ai_reviews.csv` is one file for all generators; we slice it by the `generator` column into a `{generator: DataFrame}` dict, then sanity-check that every generator is present with a reasonable count.

In [ ]:
import pandas as pd

human = pd.read_csv(HUMAN_PATH)
ai = pd.read_csv(AI_PATH)

print(f'human: {len(human)} reviews')
print(f'ai: {len(ai)} reviews — by generator:')
print(ai['generator'].value_counts())

gens = {g: ai[ai['generator'] == g].reset_index(drop=True) for g in GENERATORS}
for g in GENERATORS:
    assert len(gens[g]) > 0, f'no reviews found for generator {g!r} in {AI_PATH}'
    if len(gens[g]) < 2000:
        print(f'WARNING: {g} has only {len(gens[g])} reviews (expected ~2500)')

## Build one rotation

Same logic as the original `src/data/build_splits.py`, kept inline so this notebook is self-contained:
1. carve the human pool — `N_CROSSGEN_HUMAN` reserved for cross-gen test, the rest for in-dist
2. 80/10/10 train/val/test split of each of the 3 in-distribution generators
3. balance the human side 1:1 against the AI counts, drawing from the in-dist human pool
4. cross-gen test = held-out generator's reviews + the reserved human pool

In [ ]:
def split_indist(df, seed):
    """80/10/10 train/val/test for one in-dist generator's reviews."""
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train, n_val = int(n * 0.8), int(n * 0.1)
    return df.iloc[:n_train], df.iloc[n_train:n_train + n_val], df.iloc[n_train + n_val:]


def build_rotation(rotation, human, gens, seed=SEED):
    held_out = GENERATORS[rotation]
    train_gens = [g for g in GENERATORS if g != held_out]
    rs = seed + rotation

    # 1) carve human pool: reserved cross-gen humans vs in-dist human pool
    human_shuffled = human.sample(frac=1.0, random_state=rs).reset_index(drop=True)
    human_crossgen = human_shuffled.iloc[:N_CROSSGEN_HUMAN]
    human_indist_pool = human_shuffled.iloc[N_CROSSGEN_HUMAN:].reset_index(drop=True)

    # 2) per-generator 80/10/10 for the 3 in-dist generators
    train_chunks, val_chunks, test_chunks = [], [], []
    for g in train_gens:
        tr, va, te = split_indist(gens[g], seed=rs)
        train_chunks.append(tr); val_chunks.append(va); test_chunks.append(te)
    ai_train = pd.concat(train_chunks, ignore_index=True)
    ai_val = pd.concat(val_chunks, ignore_index=True)
    ai_test = pd.concat(test_chunks, ignore_index=True)

    # 3) balance human 1:1 against AI counts, from the in-dist human pool
    h_pool = human_indist_pool.sample(frac=1.0, random_state=rs).reset_index(drop=True)
    n_tr, n_va, n_te = len(ai_train), len(ai_val), len(ai_test)
    assert n_tr + n_va + n_te <= len(h_pool), (
        f'rotation {rotation}: need {n_tr + n_va + n_te} human reviews to balance '
        f'but in-dist pool only has {len(h_pool)}'
    )
    h_train = h_pool.iloc[:n_tr]
    h_val = h_pool.iloc[n_tr:n_tr + n_va]
    h_test = h_pool.iloc[n_tr + n_va:n_tr + n_va + n_te]

    # 4) cross-gen test = held-out generator + reserved human pool
    ai_crossgen = gens[held_out]

    def combine(*dfs):
        return pd.concat(dfs, ignore_index=True).sample(frac=1.0, random_state=rs).reset_index(drop=True)

    return {
        'train': combine(h_train, ai_train),
        'val_indist': combine(h_val, ai_val),
        'test_indist': combine(h_test, ai_test),
        'test_crossgen': combine(human_crossgen, ai_crossgen),
        'meta': pd.DataFrame([{
            'rotation': rotation,
            'held_out': held_out,
            'train_gens': ','.join(train_gens),
            'n_train': n_tr + len(h_train),
            'n_val': n_va + len(h_val),
            'n_test_indist': n_te + len(h_test),
            'n_test_crossgen': len(ai_crossgen) + len(human_crossgen),
        }]),
    }

## Build all 4 rotations, write CSVs, and check for leakage

In [ ]:
from pathlib import Path

for r in range(len(GENERATORS)):
    splits = build_rotation(r, human, gens)
    rdir = Path(SPLITS_DIR) / f'rotation_{r}'
    rdir.mkdir(parents=True, exist_ok=True)
    for name, df in splits.items():
        df.to_csv(rdir / f'{name}.csv', index=False)

    m = splits['meta'].iloc[0]
    print(f"rotation {r} (held out: {m['held_out']}) — "
          f"train {m['n_train']}, val {m['n_val']}, "
          f"test_indist {m['n_test_indist']}, test_crossgen {m['n_test_crossgen']}")

    # leakage guard: no review_id may appear in both train and the cross-gen test set
    train_ids = set(splits['train']['review_id'])
    cg_ids = set(splits['test_crossgen']['review_id'])
    assert train_ids.isdisjoint(cg_ids), f'LEAKAGE in rotation {r}: train and cross-gen share review_ids'
    print('  no review_id leakage between train and cross-gen test')

print(f'\nall rotations written to {SPLITS_DIR}')

## Sanity check — label balance per split

In [ ]:
for r in range(len(GENERATORS)):
    rdir = Path(SPLITS_DIR) / f'rotation_{r}'
    print(f'rotation {r}:')
    for name in ['train', 'val_indist', 'test_indist', 'test_crossgen']:
        df = pd.read_csv(rdir / f'{name}.csv')
        counts = df['label'].value_counts().to_dict()
        print(f'  {name:15s} n={len(df):6d}  label counts (0=human,1=AI)={counts}')